# Used Car Pricing: Profit Maximization (HW 4)

**Goal:** Set OfferPrice and BestGuessAtPrice for 8,531 cars to maximize dealer profit.
- If Offer ≥ 85% of true value → we buy; profit = TrueValue − Offer − $75.
- If Offer < 85% → no deal; profit = $0.
- **Target:** Expected profit ≥ $608,680.76 on template rows.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Load data and split train / template (evaluation) cars

In [2]:
df = pd.read_csv('Cars_HW_data.csv')
template = pd.read_excel('Cars_HW_template.xlsx')

unlabeled_ids = set(template['ID'])
train_full = df[~df['ID'].isin(unlabeled_ids)].copy()
test_submit = df[df['ID'].isin(unlabeled_ids)].copy()
test_submit = test_submit.set_index('ID').reindex(template['ID']).reset_index()

train_full['Car_Age'] = 2026 - train_full['Year']
test_submit['Car_Age'] = 2026 - test_submit['Year']

drop_cols = ['ID', 'Price', 'Year']
if 'OfferPrice' in train_full.columns: drop_cols.append('OfferPrice')
if 'BestGuessAtPrice' in train_full.columns: drop_cols.append('BestGuessAtPrice')

X_full = train_full.drop(columns=drop_cols)
y_full = train_full['Price']

num_cols = X_full.select_dtypes(include=['int64','float64']).columns
cat_cols = X_full.select_dtypes(include=['object','bool','string']).columns
for c in cat_cols:
    X_full[c] = X_full[c].astype(str)
    test_submit[c] = test_submit[c].astype(str)

print(f'Training: {X_full.shape[0]:,} | Deployment: {len(test_submit):,}')

# Dataset check summary
print('\n--- Dataset check ---')
print(f'Cars_HW_data: {len(df):,} rows, {len(df.columns)} cols')
print(f'Template (evaluation) IDs: {len(template):,}')
print(f'Train (not in template): {len(train_full):,}  |  Test (in template): {len(test_submit):,}')
print(f'Price (train) — min: ${y_full.min():,.0f}, max: ${y_full.max():,.0f}, mean: ${y_full.mean():,.0f}')
missing = X_full.isnull().sum()
if missing.any():
    print('Missing counts (non-zero):', missing[missing > 0].to_dict())
else:
    print('Missing values: none')

Training: 30,000 | Deployment: 8,531

--- Dataset check ---
Cars_HW_data: 38,531 rows, 28 cols
Template (evaluation) IDs: 8,531
Train (not in template): 30,000  |  Test (in template): 8,531
Price (train) — min: $1, max: $50,000, mean: $6,663
Missing counts (non-zero): {'Engine_Capacity': 4}


## 2. Preprocessor and train/validation split

In [3]:
num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                           ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', num_transformer, num_cols), ('cat', cat_transformer, cat_cols)])

X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

## 3. Train XGBoost, sweep alpha for profit, then retrain and export

In [ ]:
model = Pipeline([('prep', preprocessor), ('reg', XGBRegressor(random_state=42, n_jobs=-1, n_estimators=1000, max_depth=8, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8))])
model.fit(X_train, y_train)
val_preds = model.predict(X_val)
print('Validation RMSE:', np.sqrt(mean_squared_error(y_val, val_preds)))

def compute_profit(actual, offer):
    actual, offer = np.array(actual), np.array(offer)
    accepted = offer >= 0.85 * actual
    profit = np.where(accepted, actual - offer - 75, 0)
    return profit.sum(), accepted.sum()

# Conservative predictions: shrinkage + alpha grid to maximize validation profit
SHRINKAGE_VALUES = [0.85, 0.88, 0.90, 0.92]
alphas = np.linspace(0.82, 0.98, 33)
best_profit = -np.inf
best_alpha = 0.85
best_shrink = 0.90
best_win_rate = 0.0
profits_by_alpha = None
win_rates_by_alpha = None

for shrink in SHRINKAGE_VALUES:
    val_preds_conservative = val_preds * shrink
    profits = []
    win_rates = []
    for a in alphas:
        p, wins = compute_profit(y_val, val_preds_conservative * a)
        profits.append(p)
        win_rates.append(wins / len(y_val))
    opt_idx = np.argmax(profits)
    if profits[opt_idx] > best_profit:
        best_profit = profits[opt_idx]
        best_alpha = alphas[opt_idx]
        best_shrink = shrink
        best_win_rate = win_rates[opt_idx]
        profits_by_alpha = np.array(profits)
        win_rates_by_alpha = np.array(win_rates)

n_test = len(test_submit)
estimated_test_profit = (best_profit / len(y_val)) * n_test
TARGET = 608_680.76
meets_target = estimated_test_profit >= TARGET
print('--- OPTIMIZATION (shrinkage + alpha) ---')
print(f'Best shrinkage: {best_shrink:.2f}  Best alpha: {best_alpha:.4f}')
print(f'Effective offer = raw_pred * {best_shrink * best_alpha:.4f}')
print(f'Expected validation profit: ${best_profit:,.2f}  (win rate: {best_win_rate*100:.1f}%)')
print(f'Scaled profit estimate (validation → template size): ${estimated_test_profit:,.2f}  ->  Target: ${TARGET:,.2f}')
print('Target met:', 'YES' if meets_target else 'NO')

model.fit(X_full, y_full)
X_test = test_submit.drop(columns=drop_cols, errors='ignore')
test_preds_raw = model.predict(X_test)
template['BestGuessAtPrice'] = test_preds_raw
template['OfferPrice'] = test_preds_raw * best_shrink * best_alpha
template.to_excel('Cars_HW_template_predictions.xlsx', index=False)
print('Exported Cars_HW_template_predictions.xlsx')